# Week 13–14: Ensemble Methods, Bias–Variance Trade-off & Regularization

**Topics Covered**
- Bagging
- Boosting
- XGBoost
- LightGBM
- Bias–Variance Trade-off
- Overfitting and Underfitting
- Regularization: L1 and L2
- Practical implementation and model comparison

**Objective:** Understand how ensemble learning improves prediction performance and how bias, variance, overfitting, underfitting, and regularization affect model generalization.

## 1. Ensemble Learning

**Ensemble learning** combines predictions from multiple machine-learning models to produce a stronger and more robust model.

### Main approaches
1. **Bagging** – trains models independently on different bootstrap samples and combines their predictions.
2. **Boosting** – trains models sequentially, where later models focus on correcting errors made by earlier models.

### Advantages
- Better predictive performance
- Improved stability
- Reduced variance or bias depending on the method
- Better generalization on unseen data

## 2. Bagging

**Bagging (Bootstrap Aggregating)** creates multiple training datasets by sampling the original dataset with replacement.

Each base model is trained independently.

For classification, predictions are usually combined by **majority voting**.  
For regression, predictions are usually combined by **averaging**.

### Example
**Random Forest** is a popular bagging-based ensemble method.

### Key idea
> Train many models independently → combine their predictions → reduce variance.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

X, y = load_breast_cancer(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=42),
    n_estimators=100,
    random_state=42
)

bagging.fit(X_train, y_train)
y_pred = bagging.predict(X_test)

print("Bagging Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## 3. Boosting

Boosting builds models **sequentially**.

Each new model attempts to improve the overall model by focusing on errors or difficult examples from previous models.

### Popular boosting algorithms
- AdaBoost
- Gradient Boosting
- **XGBoost**
- **LightGBM**

### Key idea
> Train weak learners sequentially → correct previous errors → create a strong learner.

## 4. XGBoost

**XGBoost (Extreme Gradient Boosting)** is an optimized gradient-boosting algorithm based on decision trees.

### Important features
- Gradient boosting with regularization
- Handles nonlinear relationships
- Supports missing values
- Efficient and scalable
- Uses tree complexity controls to reduce overfitting

### Important parameters
- `n_estimators` – number of boosting rounds
- `max_depth` – maximum tree depth
- `learning_rate` – contribution of each tree
- `subsample` – fraction of training samples used
- `colsample_bytree` – fraction of features used
- `reg_alpha` – L1 regularization
- `reg_lambda` – L2 regularization

In [ ]:
# Install if XGBoost is not already available:
# %pip install xgboost

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

## 5. LightGBM

**LightGBM** is a gradient-boosting framework designed for high efficiency and large datasets.

It uses a **leaf-wise tree growth strategy**, which can achieve high accuracy with fewer trees, but may overfit on small datasets if not controlled.

### Important parameters
- `n_estimators`
- `learning_rate`
- `num_leaves`
- `max_depth`
- `min_child_samples`
- `subsample`
- `colsample_bytree`
- `reg_alpha`
- `reg_lambda`

### XGBoost vs LightGBM

| Feature | XGBoost | LightGBM |
|---|---|---|
| Boosting | Gradient boosting | Gradient boosting |
| Tree growth | Level-wise/depth-wise | Leaf-wise |
| Speed | Fast | Often very fast |
| Large datasets | Good | Excellent |
| Main risk | Overfitting with complex trees | Overfitting with too many leaves |
| Regularization | Strong | Strong |

In [ ]:
# Install if LightGBM is not already available:
# %pip install lightgbm

from lightgbm import LGBMClassifier

lgbm_model = LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    verbosity=-1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)

print("LightGBM Accuracy:", accuracy_score(y_test, lgbm_pred))

In [ ]:
# Compare ensemble models

models = {
    "Bagging": bagging,
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb_model,
    "LightGBM": lgbm_model
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, pred)

for name, score in results.items():
    print(f"{name:15s}: {score:.4f}")

## 6. Bias–Variance Trade-off

Prediction error can be understood using **bias** and **variance**.

### Bias
Bias is error caused by overly simple assumptions in the model.

- High bias → model is too simple
- Usually associated with **underfitting**

### Variance
Variance describes how much a model's prediction changes when trained on different datasets.

- High variance → model is too sensitive to training data
- Usually associated with **overfitting**

### Trade-off

A good model balances bias and variance.

- **High bias + low variance → underfitting**
- **Low bias + high variance → overfitting**
- **Reasonably low bias + reasonably low variance → good generalization**

A useful conceptual relationship is:

**Expected Test Error ≈ Bias² + Variance + Irreducible Error**

In [ ]:
# Simple visualization of the bias-variance idea

import numpy as np
import matplotlib.pyplot as plt

complexity = np.arange(1, 11)

bias_squared = (11 - complexity) ** 2 / 100
variance = complexity ** 2 / 100
total_error = bias_squared + variance

plt.figure(figsize=(9, 5))
plt.plot(complexity, bias_squared, marker="o", label="Bias²")
plt.plot(complexity, variance, marker="o", label="Variance")
plt.plot(complexity, total_error, marker="o", label="Total Error")
plt.xlabel("Model Complexity")
plt.ylabel("Error")
plt.title("Bias–Variance Trade-off")
plt.legend()
plt.grid(True)
plt.show()

## 7. Overfitting

**Overfitting** occurs when a model learns the training data too closely, including noise and irrelevant patterns.

### Signs
- Very high training accuracy
- Much lower validation/test accuracy
- Large train–test performance gap

### Causes
- Model is too complex
- Too many parameters
- Very deep decision trees
- Too little training data
- Noise in the dataset
- Insufficient regularization

### Prevention
- Use cross-validation
- Reduce model complexity
- Apply L1/L2 regularization
- Use pruning or early stopping
- Increase training data
- Feature selection
- Use ensemble methods appropriately

## 8. Underfitting

**Underfitting** occurs when a model is too simple to learn the important patterns in the data.

### Signs
- Low training accuracy
- Low validation/test accuracy
- Both training and test errors are high

### Causes
- Model is too simple
- Excessive regularization
- Too few features
- Insufficient training
- Poor feature representation

### Prevention
- Increase model complexity
- Add useful features
- Reduce excessive regularization
- Train for longer when appropriate
- Use a more suitable algorithm

In [ ]:
# Demonstrating underfitting vs overfitting with decision trees

from sklearn.tree import DecisionTreeClassifier

depths = [1, 2, 4, 8, 15, None]

print("Depth | Train Accuracy | Test Accuracy")
print("-" * 42)

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)

    train_acc = tree.score(X_train, y_train)
    test_acc = tree.score(X_test, y_test)

    print(f"{str(depth):>5} | {train_acc:14.4f} | {test_acc:13.4f}")

## 9. Regularization

Regularization adds a penalty to the learning objective to discourage overly complex models.

It helps:
- Reduce overfitting
- Control model complexity
- Improve generalization
- Stabilize model parameters

### L1 Regularization

L1 adds the absolute value of model coefficients to the loss.

**Penalty:** `λ Σ |wᵢ|`

Main effect:
- Can force some coefficients to exactly zero
- Useful for feature selection
- Produces sparse models

### L2 Regularization

L2 adds the squared value of model coefficients.

**Penalty:** `λ Σ wᵢ²`

Main effect:
- Shrinks coefficients toward zero
- Usually does not make them exactly zero
- Helps stabilize the model

### Comparison

| Property | L1 | L2 |
|---|---|---|
| Penalty | Absolute coefficients | Squared coefficients |
| Feature selection | Yes, often | No, generally |
| Sparse model | Yes | Usually no |
| Main effect | Shrinks some weights to zero | Smoothly shrinks weights |
| Common name | Lasso | Ridge |

In [ ]:
# Practical L1 vs L2 regularization example

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

l1_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l1", solver="liblinear", C=1.0, max_iter=2000
    ))
])

l2_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        penalty="l2", solver="lbfgs", C=1.0, max_iter=2000
    ))
])

l1_model.fit(X_train, y_train)
l2_model.fit(X_train, y_train)

print("L1 Test Accuracy:", l1_model.score(X_test, y_test))
print("L2 Test Accuracy:", l2_model.score(X_test, y_test))

l1_coef = l1_model.named_steps["classifier"].coef_[0]
l2_coef = l2_model.named_steps["classifier"].coef_[0]

print("Number of zero coefficients with L1:", np.sum(l1_coef == 0))
print("Number of zero coefficients with L2:", np.sum(np.isclose(l2_coef, 0)))

## 10. Important Parameters for Controlling Overfitting

### In XGBoost
- Lower `max_depth`
- Lower `learning_rate`
- Use `subsample`
- Use `colsample_bytree`
- Increase `reg_alpha` for L1 regularization
- Increase `reg_lambda` for L2 regularization
- Use early stopping when validation data is available

### In LightGBM
- Control `num_leaves`
- Set an appropriate `max_depth`
- Increase `min_child_samples`
- Use `subsample` and `colsample_bytree`
- Use `reg_alpha` and `reg_lambda`
- Use early stopping

**Important:** Regularization strength should be tuned using validation/cross-validation rather than chosen arbitrarily.

## 11. Summary

| Topic | Main Idea |
|---|---|
| Bagging | Train models independently and aggregate predictions |
| Boosting | Train models sequentially to correct errors |
| XGBoost | Efficient, regularized gradient boosting |
| LightGBM | Fast gradient boosting using leaf-wise tree growth |
| Bias | Error from overly simple assumptions |
| Variance | Sensitivity to training data |
| Overfitting | Excellent training performance but poor generalization |
| Underfitting | Poor performance on both training and unseen data |
| L1 | Encourages sparse coefficients and feature selection |
| L2 | Shrinks coefficients and controls model complexity |

### Key takeaway
**The goal is not to make a model as complex as possible. The goal is to find the right level of complexity that generalizes well to unseen data.**

## 12. Viva / Exam Questions

1. What is ensemble learning?
2. Explain Bagging with an example.
3. What is the difference between Bagging and Boosting?
4. What is XGBoost?
5. What is LightGBM?
6. Why can LightGBM overfit on small datasets?
7. Define bias and variance.
8. Explain the bias–variance trade-off.
9. What is overfitting? How can it be detected?
10. What is underfitting? How can it be prevented?
11. What is regularization?
12. Differentiate between L1 and L2 regularization.
13. What is the role of `learning_rate` in boosting?
14. How can XGBoost and LightGBM be regularized?
15. Why is cross-validation useful for controlling overfitting?